In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/india_agriculture_seed_sales_data.csv')

 Farmer Sentiment Drivers

In [2]:
from scipy.stats import pearsonr

sentiment_drivers = {}
drivers = ['Seed_Quality_Index_%','Pest_Infestation_Index',
           'Irrigation_Access_%','Rainfall_mm',
           'Competitor_Market_Share_%','Farmer_Share_%']

for col in drivers:
    corr, p = pearsonr(df[col], df['Farmer_Sentiment_Score'])
    sentiment_drivers[col] = {'correlation': round(corr,3), 'p_value': round(p,4)}

print("Farmer Sentiment Correlation Analysis:")
print(pd.DataFrame(sentiment_drivers).T.sort_values('correlation', ascending=False))

Farmer Sentiment Correlation Analysis:
                           correlation  p_value
Irrigation_Access_%              0.011   0.1328
Seed_Quality_Index_%             0.009   0.2282
Farmer_Share_%                   0.002   0.8049
Competitor_Market_Share_%        0.002   0.7799
Pest_Infestation_Index          -0.004   0.5353
Rainfall_mm                     -0.005   0.5042


In [7]:
# ── Farmer Sentiment Trend ───────────────────────────────────────────────────
sent_yr = df.groupby(['Year', 'Region'])['Farmer_Sentiment_Score'].mean().reset_index()
print(sent_yr)

    Year   Region  Farmer_Sentiment_Score
0   2021  Central                6.717664
1   2021     East                6.759953
2   2021    North                6.808632
3   2021    South                6.743256
4   2021     West                6.729094
5   2022  Central                6.805398
6   2022     East                6.710614
7   2022    North                6.716890
8   2022    South                6.785557
9   2022     West                6.726366
10  2023  Central                6.703049
11  2023     East                6.799126
12  2023    North                6.795809
13  2023    South                6.791553
14  2023     West                6.791552
15  2024  Central                6.760029
16  2024     East                6.713950
17  2024    North                6.799969
18  2024    South                6.705023
19  2024     West                6.789615
20  2025  Central                6.827166
21  2025     East                6.629446
22  2025    North                6

Sentiment by Region & Season

In [8]:
season_map = {12:'Winter',1:'Winter',2:'Winter',
              3:'Summer',4:'Summer',5:'Summer',
              6:'Monsoon',7:'Monsoon',8:'Monsoon',
              9:'Post-Monsoon',10:'Post-Monsoon',11:'Post-Monsoon'}
df['Season'] = df['Month'].map(season_map)

sentiment_season = df.groupby(['Region','Season'])['Farmer_Sentiment_Score'].mean().round(2)
print("\nFarmer Sentiment by Region & Season:")
print(sentiment_season.unstack())


Farmer Sentiment by Region & Season:
Season   Monsoon  Post-Monsoon  Summer  Winter
Region                                        
Central     6.80          6.76    6.72    6.74
East        6.79          6.66    6.76    6.72
North       6.77          6.74    6.81    6.75
South       6.76          6.73    6.77    6.74
West        6.79          6.72    6.73    6.81


Quality-Sales Relationship

In [9]:
quality_bins = pd.cut(df['Seed_Quality_Index_%'],
    bins=[60,75,85,95,100], labels=['Poor (60-75)','Fair (75-85)','Good (85-95)','Excellent (95+)'])
quality_impact = df.groupby(quality_bins, observed=True).agg(
    Avg_Units_Sold=('Actual_Units_Sold','mean'),
    Avg_Margin=('Profit_Margin_%','mean'),
    Avg_Sentiment=('Farmer_Sentiment_Score','mean')
).round(2)
print("\nSeed Quality vs Business Outcomes:")
print(quality_impact)


Seed Quality vs Business Outcomes:
                      Avg_Units_Sold  Avg_Margin  Avg_Sentiment
Seed_Quality_Index_%                                           
Fair (75-85)                  953.06        5.87           6.76
Good (85-95)                  950.46        6.14           6.75
Excellent (95+)               946.77        6.18           6.77
